# [8.4] Circuit Tracing with Attribution Graphs - Exercises

Build a local attribution-graph harness: edge scores, top-k directed graphs, target-metric explanation, path perturbations, alternative baselines, and counterfactual summary checks.

In [ ]:
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import torch as t

chapter = "chapter8_automated_circuits"
section = "part4_circuit_tracing_attribution_graphs"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part4_circuit_tracing_attribution_graphs.tests as tests

GT_TIER = "GT-1"
EXERCISE_ID = "8.4.circuit_tracing_attribution_graphs"
EXPECTED_RUNTIME = "35-50 minutes for exercises; about 1-2 minutes for CUDA preflight"
REQUIRES_GPU = False
CounterfactualDirection = Literal["increase", "decrease"]

In [ ]:
@dataclass(frozen=True)
class CircuitTraceEdge:
    source: str
    target: str
    score: float


@dataclass(frozen=True)
class LocalAttributionGraph:
    nodes: tuple[str, ...]
    edges: tuple[CircuitTraceEdge, ...]


@dataclass(frozen=True)
class GraphMetricReport:
    full_metric: float
    corrupt_metric: float
    graph_metric: float
    explained_fraction: float
    explains_target_metric: bool


@dataclass(frozen=True)
class PathPerturbationReport:
    original_metric: float
    perturbed_metric: float
    metric_drop: float
    top_path_survives_test: bool


@dataclass(frozen=True)
class AlternativeGraphBaselineReport:
    graph_metric: float
    alternative_metric: float
    margin: float
    alternative_baseline_fails: bool


@dataclass(frozen=True)
class GraphSummaryCounterfactualReport:
    predicted_direction: CounterfactualDirection
    observed_delta: float
    predicts_counterfactual: bool

@dataclass(frozen=True)
class AttributionPathReport:
    source: str
    target: str
    path: tuple[str, ...]
    edge_scores: tuple[float, ...]
    path_score: float
    reaches_target: bool

## Edge Scores

Compute source-by-target scores from upstream activation deltas and downstream gradients.

In [ ]:
def edge_attribution_scores(
    upstream_activation_delta: t.Tensor,
    downstream_gradients: t.Tensor,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_edge_attribution_scores_forms_position_edge_matrix(
    edge_attribution_scores,
)

## Local Attribution Graphs

Build a directed top-k graph while keeping source and target names visible.

In [ ]:
def build_local_attribution_graph(
    edge_scores: t.Tensor,
    node_names: list[str],
    *,
    top_k: int = 3,
) -> LocalAttributionGraph:
    raise NotImplementedError()


tests.test_build_local_attribution_graph_keeps_top_directed_edges(
    build_local_attribution_graph,
)

## Target Metric Explanation

Normalize graph behavior by the clean-corrupt gap.

In [ ]:
def graph_metric_report(
    *,
    full_metric: float,
    corrupt_metric: float,
    graph_metric: float,
    min_explained_fraction: float = 0.75,
) -> GraphMetricReport:
    raise NotImplementedError()


tests.test_graph_metric_report_measures_explained_fraction(graph_metric_report)

## Perturbation And Baselines

A graph needs causal damage under perturbation and a worse plausible alternative.

In [ ]:
def path_perturbation_report(
    *,
    original_metric: float,
    perturbed_metric: float,
    min_metric_drop: float = 0.5,
) -> PathPerturbationReport:
    raise NotImplementedError()


def alternative_graph_baseline_report(
    *,
    graph_metric: float,
    alternative_metric: float,
    min_margin: float = 0.5,
) -> AlternativeGraphBaselineReport:
    raise NotImplementedError()


tests.test_path_perturbation_and_alternative_baseline_reports(
    path_perturbation_report,
    alternative_graph_baseline_report,
)

## Counterfactual Summaries

Make the graph story falsifiable by predicting a direction before measuring the intervention.

In [ ]:
def graph_summary_counterfactual_report(
    *,
    predicted_direction: CounterfactualDirection,
    baseline_metric: float,
    counterfactual_metric: float,
) -> GraphSummaryCounterfactualReport:
    raise NotImplementedError()


tests.test_counterfactual_summary_report_checks_direction(
    graph_summary_counterfactual_report,
)

## Multi-Hop Attribution Paths

A useful attribution graph should expose a directed causal chain, not only a single large edge. Find the strongest path from a source feature to the target logit and explicitly report when no directed path exists.

In [ ]:
def top_attribution_path(
    graph: LocalAttributionGraph,
    *,
    source: str,
    target: str,
    max_depth: int = 4,
) -> AttributionPathReport:
    raise NotImplementedError()


tests.test_top_attribution_path_recovers_multi_hop_chain(top_attribution_path)

## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
